In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
import sys
DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk_data\studies\tropoflavin_nootropics\runs\2026-08-31-comparator-cohort\sentiment\comparators.db"
STUDY_DIR = r"C:\Users\scgee\AppData\Local\Temp\claude\C--Users-scgee-OneDrive-Documents-Projects-PatientPunk\3e835e38-efdc-47a2-bdd4-29ef8f2558d6\scratchpad\wt-140\studies\tropoflavin_nootropics"
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(DB_PATH)
sys.path.insert(0, STUDY_DIR)
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score.

    Unrecognized/missing labels return NaN (excluded from means) rather than
    0.0, to avoid silently biasing averages toward neutral.
    """
    return SENTIMENT_SCORE.get(s, np.nan)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score is None or pd.isna(avg_score):
        return None
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if pd.isna(k) or pd.isna(n) or n <= 0 or k < 0 or k > n:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0.0, center - margin), min(1.0, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal, inverted, or invalid."""
    if pd.isna(treatment_rate) or pd.isna(baseline_rate):
        return None
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors — include both aggregated ("mixed/neutral") and raw DB labels
# ("mixed", "neutral") so charts can key off either without a KeyError.
COLORS = {
    "positive": "#2ecc71",
    "mixed/neutral": "#95a5a6",
    "mixed": "#95a5a6",
    "neutral": "#95a5a6",
    "negative": "#e74c3c",
}


**Research question:** *Does the dose someone takes, or the route they take it by, predict whether they report a side effect?*

# Dose, Route and Side-Effect Reporting in Tropoflavin Users

## Abstract

Pipeline B records the dose and administration route a patient states; Pipeline A records
whether they named a side effect. The two share an author key, so they can be joined to
ask whether dose or route predicts reported harm.

The answer is **no**, and the more useful finding is *why the question is hard to answer*.
Across 159 observed (author, compound) exposures, only 64 carry a dose and 62 a route.
Every dose band's 95% interval contains the cohort baseline; a trend test across the six
bands is non-significant, and so is sublingual against every other route.

This is the second build. The first rested on an extraction that had silently missed a
large share of the compound mentions in its own corpus — see issue #143. A repair run
recovered 25 exposures, raising the dosed sample by 28% and the routed sample by 35%.
**The conclusion did not change.** The added data narrowed the mean 95% interval from 59
to 55 percentage points and moved nothing off the baseline. That is a modest gain, and
saying so is the point: this is a better-supported null, not a new result.


## 1. Where the numbers come from

| quantity | source |
|---|---|
| dose, administration route | Pipeline B — `pipeline_b_compound_exposures`, plus the 2026-09-01 repair run |
| side-effect presence | Pipeline A — `treatment_reports.side_effects` in `comparators.db` |
| join key | `author_hash` = `user_id` (13,545 of 13,568 linked authors resolve) |

`comparators.db` keeps 7,8-DHF and 4′-DMA as separate drug ids, so side effects attribute
per compound. The linked database merges them under one `treatment` row and cannot.

**The outcome is a reporting rate, not clinical incidence.** It means "this author named a
side effect somewhere in their posts". Silence is not tolerance — it is usually a post
about something else.

In [ ]:
import dose_route_se_data as D
import collections

# The setup cell opened comparators.db alone. The merge also reads the linked study
# database, so let the module open both -- it attaches the linked db as schema `L`.
conn = D.connect()

rows = D.build(conn)
obs  = [r for r in rows if r["observed"]]

origin = collections.Counter(r["origin"] for r in rows)
print(f"(author, compound) exposures : {len(rows)}")
for k, v in origin.most_common():
    print(f"   {k:22} {v}")
print(f"\nobserved (author has a sentiment report for that compound): {len(obs)}")
print(f"   ...with a dose band : {sum(1 for r in obs if r['dose_band'])}")
print(f"   ...with a route     : {sum(1 for r in obs if r['route'])}")

k = sum(r["has_se"] for r in obs)
p, lo, hi = D.wilson(k, len(obs))
print(f"\nBASELINE side-effect reporting rate: {k}/{len(obs)} = {100*p:.1f}%"
      f"  [{100*lo:.1f}%, {100*hi:.1f}%]")

### The corroboration filter

The repair raised recall by telling the model to extract complete stack lists. That cost
attribution accuracy: it began pairing a compound with whatever dose sat nearby. A
recovered dose is therefore kept only when the number actually appears within 400
characters of a mention of the compound it was attached to.

Three doses fail that check and are dropped here. All three were verified by hand as
belonging to a different compound in the same passage — 100 mg of Phenyl Hydrazide,
10 mg of Kratom, and "less than 5 mg" of a tryptamine.

In [ ]:
import csv

dropped = []
raw_pairs = kept_pairs = 0
for row in csv.DictReader((D.REPAIR / "improved" / "records.csv").open(encoding="utf-8")):
    text = None
    for item in (row.get("dosage") or "").split("|"):
        if ":" not in item:
            continue
        treatment, value = (s.strip() for s in item.split(":", 1))
        if not D.TARGET.search(treatment) or D.to_mg(value) is None:
            continue
        raw_pairs += 1
        if text is None:
            text = D.author_text(row["author_hash"])
        if D.corroborated(text, value):
            kept_pairs += 1
        else:
            dropped.append((row["author_hash"][:8], treatment, value))

print(f"numeric target doses extracted : {raw_pairs}")
print(f"corroborated, kept             : {kept_pairs}")
print(f"dropped as unattributable      : {len(dropped)}")
for a, t, v in dropped:
    print(f"   [{a}] {t} -> {v}")

## 2. Side-effect reporting by dose band

Each row is a dose band. The dot is the point estimate, the bar the 95% Wilson interval,
and the dashed line the cohort baseline. Wilson rather than the normal approximation
because most cells hold fewer than fifteen exposures.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def rate_table(subset, key, order):
    groups = collections.defaultdict(list)
    for r in subset:
        if r[key]:
            groups[r[key]].append(r)
    out = []
    for label in order:
        g = groups.get(label)
        if not g:
            continue
        k, n = sum(x["has_se"] for x in g), len(g)
        p, lo, hi = D.wilson(k, n)
        out.append(dict(label=label, n=n, k=k, p=p, lo=lo, hi=hi))
    return out

def forest(table, title, baseline, ax):
    y = np.arange(len(table))[::-1]
    ax.axvline(100 * baseline, ls="--", lw=1, color="#888", zorder=1)
    # Direct-label the reference line: a legend box collides with the widest bar.
    # Blended transform -- x in data units, y as a fraction of the axes.
    ax.text(100 * baseline + 1.5, 0.985, f"cohort baseline {100*baseline:.1f}%",
            transform=ax.get_xaxis_transform(), va="top", ha="left",
            fontsize=8, color="#666")
    for yi, row in zip(y, table):
        ax.plot([100 * row["lo"], 100 * row["hi"]], [yi, yi], lw=6,
                color="#2a78d6", alpha=0.25, solid_capstyle="butt", zorder=2)
        ax.plot(100 * row["p"], yi, "o", ms=7, color="#2a78d6", zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['label']}  (n={r['n']})" for r in table])
    ax.set_xlim(0, 100); ax.set_xlabel("% reporting >=1 side effect")
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.grid(axis="x", lw=0.5, alpha=0.3); ax.set_axisbelow(True)
    for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
    ax.margins(y=0.08)

base = sum(r["has_se"] for r in obs) / len(obs)
dose_tbl = rate_table(obs, "dose_band", D.BAND_ORDER)

fig, ax = plt.subplots(figsize=(9, 0.5 * len(dose_tbl) + 2))
forest(dose_tbl, "Reported >=1 side effect, by dose band (both tropoflavin compounds)",
       base, ax)
plt.tight_layout(); plt.show()

print(f"{'band':16}{'n':>5}{'>=1 SE':>8}{'rate':>8}   95% CI")
for r in dose_tbl:
    print(f"{r['label']:16}{r['n']:>5}{r['k']:>8}{100*r['p']:>7.1f}%   "
          f"[{100*r['lo']:>4.0f}%, {100*r['hi']:>4.0f}%]")

### Does the rate trend with dose?

A Cochran–Armitage test across the six ordered milligram bands. Bands are compared
**within compound** where possible: 4′-DMA is roughly an order of magnitude more potent,
so pooling raw milligrams across the two would be meaningless.

In [ ]:
quant = [r for r in obs if r["dose_band"] in D.BAND_ORDER[:6]]
cells = []
for i, band in enumerate(D.BAND_ORDER[:6], start=1):
    g = [r for r in quant if r["dose_band"] == band]
    if g:
        cells.append((i, sum(r["has_se"] for r in g), len(g)))
z, p = D.cochran_armitage(cells)
print(f"all tropoflavin exposures : z = {z:+.2f}, p = {p:.3f}   (n = {sum(c[2] for c in cells)})")

for compound in ("7,8-DHF", "4'-DMA"):
    sub = [r for r in quant if r["compound"] == compound]
    cc = []
    for i, band in enumerate(D.BAND_ORDER[:6], start=1):
        g = [r for r in sub if r["dose_band"] == band]
        if g:
            cc.append((i, sum(r["has_se"] for r in g), len(g)))
    if sum(c[2] for c in cc) >= 8:
        z2, p2 = D.cochran_armitage(cc)
        print(f"{compound:25} : z = {z2:+.2f}, p = {p2:.3f}   (n = {sum(c[2] for c in cc)})")

lower = [r for r in quant if (r["dose_order"] or 9) <= 3]
upper = [r for r in quant if (r["dose_order"] or 0) >= 4]
a, b = sum(r["has_se"] for r in upper), len(upper) - sum(r["has_se"] for r in upper)
c_, d_ = sum(r["has_se"] for r in lower), len(lower) - sum(r["has_se"] for r in lower)
print(f"\n<25 mg vs >=25 mg : {c_}/{len(lower)} vs {a}/{len(upper)}, "
      f"Fisher exact p = {D.fisher_exact(a, b, c_, d_):.3f}")

## 3. Side-effect reporting by route

Sublingual (`oral mucosal`) dominates this cohort. The recovered exposures widen the
comparison group, which is where the repair helped most.

In [ ]:
route_tbl = rate_table(obs, "route", D.ROUTE_ORDER)
fig, ax = plt.subplots(figsize=(9, 0.5 * len(route_tbl) + 2))
forest(route_tbl, "Reported >=1 side effect, by administration route", base, ax)
plt.tight_layout(); plt.show()

print(f"{'route':26}{'n':>5}{'>=1 SE':>8}{'rate':>8}   95% CI")
for r in route_tbl:
    print(f"{r['label']:26}{r['n']:>5}{r['k']:>8}{100*r['p']:>7.1f}%   "
          f"[{100*r['lo']:>4.0f}%, {100*r['hi']:>4.0f}%]")

subl = [r for r in obs if r["route"] == "oral mucosal"]
other = [r for r in obs if r["route"] and r["route"] != "oral mucosal"]
a, b = sum(r["has_se"] for r in subl), len(subl) - sum(r["has_se"] for r in subl)
c_, d_ = sum(r["has_se"] for r in other), len(other) - sum(r["has_se"] for r in other)
print(f"\nsublingual {a}/{len(subl)} vs every other known route {c_}/{len(other)}"
      f"  ->  Fisher exact p = {D.fisher_exact(a, b, c_, d_):.3f}")

## 4. What the recall repair changed

The honest test of the repair is not whether it produced more rows, but whether it moved
any conclusion. It did not — it narrowed intervals.

In [ ]:
orig = [r for r in obs if r["origin"] == "original"]
print(f"{'':34}{'original only':>15}{'with recovered':>16}")
for label, key, order in (("exposures with a dose", "dose_band", D.BAND_ORDER),
                          ("exposures with a route", "route", D.ROUTE_ORDER)):
    a = sum(1 for r in orig if r[key])
    b = sum(1 for r in obs if r[key])
    print(f"  {label:32}{a:>15}{b:>16}   ({100*(b-a)/max(a,1):+.0f}%)")

for label, subset in (("original only", orig), ("with recovered", obs)):
    t = rate_table(subset, "dose_band", D.BAND_ORDER)
    widths = [100 * (r["hi"] - r["lo"]) for r in t]
    print(f"\n{label}: {len(t)} dose bands, mean 95% CI width "
          f"{sum(widths)/len(widths):.0f} percentage points")

## 5. What constrains this

**1. It is a reporting rate.** "Named a side effect somewhere" is not incidence. An
author who tolerated a compound perfectly and an author who never discussed side effects
are indistinguishable here.

**2. Dose and route are rarely co-reported.** Even after the repair, only a minority of
exposures carry both, so a dose × route interaction is not estimable. The bands are
marginal, not joint.

**3. The recovered rows are ~85–90% precise.** The corroboration filter removes the three
dose misattributions it can detect. Routes are weaker: of 20 recovered route values, 4
were unsupported by any route language near the compound, including one `sublingual`
appearing nowhere in the source. Route findings should be leaned on less than dose.

**4. Side-effect strings are uncanonicalised.** Presence/absence is safe; counts are not.
`comparators.db` stores raw strings, so `headache` and `headaches` both score.

**5. Absence of an effect is not evidence of absence.** With cells this size the study is
underpowered for anything but a large effect. A true difference of ten or fifteen
percentage points between bands would not reliably show up here.

In [ ]:
from IPython.display import HTML, display
display(HTML(
    '<div style="font-size:1.05em;font-style:italic;text-align:center;padding:18px;'
    'margin-top:18px;border-top:2px solid #ccc;"><strong>These findings reflect reporting '
    'patterns in an online community, not population-level treatment effects. '
    'This is not medical advice.</strong></div>'))